# CS533 — Evaluating CoT Supervision for VQA on CLEVR
### Results Analysis Notebook

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
with open('../results/answer_only_results.json') as f:
    baseline = json.load(f)
with open('../results/chain_of_thought_results.json') as f:
    cot = json.load(f)

df_base = pd.DataFrame(baseline['results'])
df_cot  = pd.DataFrame(cot['results'])

print(f"Baseline accuracy:  {baseline['accuracy']:.4f}")
print(f"CoT model accuracy: {cot['accuracy']:.4f}")

## 1. Overall Accuracy

In [ ]:
labels = ['Baseline\n(Answer-Only)', 'CoT Model\n(Chain-of-Thought)']
accs   = [baseline['accuracy'], cot['accuracy']]
colors = ['#4C72B0', '#DD8452']

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(labels, accs, color=colors, width=0.4, edgecolor='black')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Overall VQA Accuracy on CLEVR', fontsize=13)
plt.tight_layout()
plt.savefig('../results/overall_accuracy.png', dpi=150)
plt.show()

## 2. Accuracy by Question Type

In [ ]:
types_b = baseline['type_accuracy']
types_c = cot['type_accuracy']
all_types = sorted(set(types_b) | set(types_c))

b_vals = [types_b.get(t, 0) for t in all_types]
c_vals = [types_c.get(t, 0) for t in all_types]
x = np.arange(len(all_types))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, b_vals, width, label='Baseline',   color='#4C72B0', edgecolor='black')
ax.bar(x + width/2, c_vals, width, label='CoT Model',  color='#DD8452', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(all_types, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy by Question Type', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../results/accuracy_by_type.png', dpi=150)
plt.show()

## 3. Accuracy by Reasoning Depth

In [ ]:
depths_b = baseline['depth_accuracy']
depths_c = cot['depth_accuracy']
all_buckets = sorted(set(depths_b) | set(depths_c))

b_vals = [depths_b.get(k, 0) for k in all_buckets]
c_vals = [depths_c.get(k, 0) for k in all_buckets]
x = np.arange(len(all_buckets))

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, b_vals, width, label='Baseline',  color='#4C72B0', edgecolor='black')
ax.bar(x + width/2, c_vals, width, label='CoT Model', color='#DD8452', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(all_buckets, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy by Reasoning Depth', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../results/accuracy_by_depth.png', dpi=150)
plt.show()

## 4. Summary Table

In [ ]:
summary = pd.read_csv('../results/summary_table.csv')
summary['delta'] = summary['cot'] - summary['baseline']
summary = summary.round(4)
summary.style.background_gradient(subset=['delta'], cmap='RdYlGn', vmin=-0.1, vmax=0.1)

## 5. Qualitative Examples (CoT Model)

In [ ]:
correct_ex   = [r for r in cot['results'] if r['correct']][:3]
incorrect_ex = [r for r in cot['results'] if not r['correct']][:3]

print('=== CORRECT ===')
for r in correct_ex:
    print(f"Q:    {r['question']}")
    print(f"GT:   {r['ground_truth']}  |  Pred: {r['prediction']}")
    print(f"Gen:  {r['generated_text'][:200]}")
    print()

print('=== INCORRECT ===')
for r in incorrect_ex:
    print(f"Q:    {r['question']}")
    print(f"GT:   {r['ground_truth']}  |  Pred: {r['prediction']}")
    print(f"Gen:  {r['generated_text'][:200]}")
    print()